In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer # 컬럼종류별로 다른전처리를 적용하는 도구
from sklearn.pipeline import Pipeline # 전처리와 모델학습을 하나의 흐름으로 묶는 도구
from sklearn.preprocessing import OneHotEncoder # 범주형값을 숫자형으로 바꿈
from sklearn.impute import SimpleImputer # 결측치를 정해진규칙으로 채워줌

from xgboost import XGBRegressor

In [2]:
plt.style.use("ggplot")

In [3]:
import matplotlib.font_manager as fm
import warnings
# 한글폰트설정 + -기호, 글꼴관련 경고 무시
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
df = pd.read_csv('jeju_bus.csv')
df_model = df.copy()

In [5]:
df.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


In [6]:
df.shape

(210457, 14)

In [7]:
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [8]:
df.isnull().sum()

id                  0
date                0
route_id            0
vh_id               0
route_nm            0
now_latitude        0
now_longitude       0
now_station         0
now_arrive_time     0
distance            0
next_station        0
next_latitude       0
next_longitude      0
next_arrive_time    0
dtype: int64

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 210457 entries, 0 to 210456
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                210457 non-null  int64  
 1   date              210457 non-null  str    
 2   route_id          210457 non-null  int64  
 3   vh_id             210457 non-null  int64  
 4   route_nm          210457 non-null  str    
 5   now_latitude      210457 non-null  float64
 6   now_longitude     210457 non-null  float64
 7   now_station       210457 non-null  str    
 8   now_arrive_time   210457 non-null  str    
 9   distance          210457 non-null  float64
 10  next_station      210457 non-null  str    
 11  next_latitude     210457 non-null  float64
 12  next_longitude    210457 non-null  float64
 13  next_arrive_time  210457 non-null  int64  
dtypes: float64(5), int64(4), str(5)
memory usage: 22.5 MB


In [10]:
df.describe()

,id,route_id,vh_id,now_latitude,now_longitude,distance,next_latitude,next_longitude,next_arrive_time
count,210457.000000,2.104570e+05,2.104570e+05,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000
mean,105228.000000,4.052491e+08,7.988694e+06,33.434528,126.603451,490.256100,33.434711,126.603687,85.380824
std,60753.847139,9.132404e+04,6.774077e+03,0.102350,0.123961,520.563932,0.102224,0.123838,85.051170
min,0.000000,4.051360e+08,7.983000e+06,33.244382,126.473300,97.000000,33.244382,126.473300,6.000000
25%,52614.000000,4.051365e+08,7.983093e+06,33.325283,126.523900,291.000000,33.325283,126.524550,44.000000
50%,105228.000000,4.053201e+08,7.983431e+06,33.484667,126.551050,384.000000,33.484860,126.551050,66.000000
75%,157842.000000,4.053201e+08,7.997041e+06,33.500197,126.650322,542.000000,33.500228,126.650322,102.000000
max,210456.000000,4.053281e+08,7.997124e+06,33.556167,126.935188,7461.000000,33.556167,126.935188,2996.000000


In [11]:
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [12]:
df.columns[1:-1]

Index(['date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude'],
      dtype='str')

In [13]:
target_col = df.columns[-1]
required_cols = df.columns[1:-1]
required_cols

Index(['date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude'],
      dtype='str')

In [14]:
def prepare_features(df_input, required_cols, target_col):
    data = df_input.copy()
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise ValueError(f'필수 컬럼이 누락되었습니다: {missing_cols}')
    
    data['date'] = pd.to_datetime(data['date']) # 날짜형 변환
    data['year'] = data['date'].dt.year
    data['month'] = data['date'].dt.month
    data['day'] = data['date'].dt.day
    data['dayofweek'] = data['date'].dt.dayofweek
    
    data['now_hour'] = (
        data['now_arrive_time'].astype(str).str.extract(r"(\d+)").astype(float) #extract목적이 한그룹만 뽑아서 변환이기때문에 괄호로 묶음
    )

    data = data.drop(columns=['date', 'now_arrive_time'])
    if target_col in data.columns:
        data = data.drop(columns=[target_col])
    
    if 'id' in data.columns:
        data = data.drop(columns=['id'])
    return data

In [15]:
X_preprocess = prepare_features(df_model,required_cols, target_col)
X_preprocess

,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,distance,next_station,next_latitude,next_longitude,year,month,day,dayofweek,now_hour
0,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,266.0,제대마을,33.457724,126.554014,2019,10,15,1,6.0
1,405136001,7997025,360-1,33.457724,126.554014,제대마을,333.0,제대아파트,33.458783,126.557353,2019,10,15,1,6.0
2,405136001,7997025,360-1,33.458783,126.557353,제대아파트,415.0,제주대학교,33.459893,126.561624,2019,10,15,1,6.0
3,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),578.0,제주여자중고등학교(아라방면),33.484860,126.542928,2019,10,15,1,6.0
4,405136001,7997025,360-1,33.485662,126.494923,도호동,374.0,은남동,33.485822,126.490897,2019,10,15,1,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,405328102,7983486,281-2,33.255783,126.577450,비석거리,528.0,삼아아파트,33.251896,126.574417,2019,10,28,0,21.0
210453,405328102,7983486,281-2,33.248595,126.568527,동문로터리,280.0,매일올레시장 7번입구,33.249753,126.565959,2019,10,28,0,21.0
210454,405328102,7983486,281-2,33.251891,126.560303,서귀포시 구 버스터미널,114.0,아랑조을거리 입구,33.251084,126.559551,2019,10,28,0,21.0
210455,405328102,7983486,281-2,33.251084,126.559551,아랑조을거리 입구,223.0,평생학습관,33.249504,126.558068,2019,10,28,0,21.0


In [ ]:
# 범주형 데이터 onehot encoding으로벡터화
categorical_features = [
    'route_nm',
    'now_station',
    'next_station'
]
# 여러 전처리 절차를 하나로 묶어줌 1. 결측치를 최빈값으로 2. onehotencoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
# 컬럼종류별로다른 전처리 적용. ('cat', 범주형전처리, 범주형컬럼)
preprocessor =  ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_features)
], remainder='passthrough') # remainder 지정해야 범주말고 나머지도 유지함. 

In [17]:
X_cat_encoded = preprocessor.fit_transform(X_preprocess)
X_cat_encoded

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3126497 stored elements and shape (210457, 732)>

In [18]:
encoded_feature_names = preprocessor.get_feature_names_out()
encoded_feature_names

array(['cat__route_nm_201-11', 'cat__route_nm_201-12',
       'cat__route_nm_201-13', 'cat__route_nm_201-14',
       'cat__route_nm_201-15', 'cat__route_nm_201-16',
       'cat__route_nm_201-17', 'cat__route_nm_201-18',
       'cat__route_nm_201-21', 'cat__route_nm_201-22',
       'cat__route_nm_201-24', 'cat__route_nm_201-26',
       'cat__route_nm_201-27', 'cat__route_nm_281-1',
       'cat__route_nm_281-2', 'cat__route_nm_360-1',
       'cat__route_nm_360-12', 'cat__route_nm_360-2',
       'cat__route_nm_360-7', 'cat__route_nm_365-21',
       'cat__route_nm_365-22', 'cat__now_station_911의원',
       'cat__now_station_LH아파트', 'cat__now_station_가마초등학교',
       'cat__now_station_가흥동', 'cat__now_station_거로 입구',
       'cat__now_station_견월교', 'cat__now_station_계룡동',
       'cat__now_station_고도농원', 'cat__now_station_고래왓',
       'cat__now_station_고망난돌입구', 'cat__now_station_고산동산(광양방면)',
       'cat__now_station_고산동산(아라방면)', 'cat__now_station_고성리 구 성산농협',
       'cat__now_station_고성리 성산농협', 

In [19]:
X_cat_encoded_df = pd.DataFrame(
    X_cat_encoded.toarray(),
    columns=encoded_feature_names
)

In [20]:
X_cat_encoded_df

,cat__route_nm_201-11,cat__route_nm_201-12,cat__route_nm_201-13,cat__route_nm_201-14,cat__route_nm_201-15,cat__route_nm_201-16,cat__route_nm_201-17,cat__route_nm_201-18,cat__route_nm_201-21,cat__route_nm_201-22,...,remainder__now_latitude,remainder__now_longitude,remainder__distance,remainder__next_latitude,remainder__next_longitude,remainder__year,remainder__month,remainder__day,remainder__dayofweek,remainder__now_hour
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.456267,126.551750,266.0,33.457724,126.554014,2019.0,10.0,15.0,1.0,6.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.457724,126.554014,333.0,33.458783,126.557353,2019.0,10.0,15.0,1.0,6.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.458783,126.557353,415.0,33.459893,126.561624,2019.0,10.0,15.0,1.0,6.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.479705,126.543811,578.0,33.484860,126.542928,2019.0,10.0,15.0,1.0,6.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.485662,126.494923,374.0,33.485822,126.490897,2019.0,10.0,15.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.255783,126.577450,528.0,33.251896,126.574417,2019.0,10.0,28.0,0.0,21.0
210453,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.248595,126.568527,280.0,33.249753,126.565959,2019.0,10.0,28.0,0.0,21.0
210454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.251891,126.560303,114.0,33.251084,126.559551,2019.0,10.0,28.0,0.0,21.0
210455,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,33.251084,126.559551,223.0,33.249504,126.558068,2019.0,10.0,28.0,0.0,21.0


In [21]:
Y = df_model[target_col]

In [22]:
X_train, X_test, Y_train, Y_test = train_test_split(X_preprocess, Y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((168365, 15), (42092, 15))

In [23]:
xgb_model = XGBRegressor()
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])
_ = model_pipeline.fit(X_train, Y_train)
print('학습완료')

학습완료


In [24]:
Y_pred = model_pipeline.predict(X_test)
mae = mean_absolute_error(Y_test, Y_pred)
print(f'MAE: {mae:.4f}')

MAE: 23.0865


In [25]:
new_data = pd.DataFrame([
    {
        'date': '2019-10-21',
        'route_id': 405136001,
        'vh_id': 7997025,
        'route_nm': '360-1',
        'now_latitude': 33.456267,
        'now_longitude': 126.551750,
        'now_station': '제주대학교입구',
        'now_arrive_time': '08시',
        'distance': 300.0,
        'next_station': '제대마을',
        'next_latitude': 33.457724,
        'next_longitude': 126.554014
    }
])

In [26]:
new_data_prepared = prepare_features(new_data, required_cols, target_col)

In [27]:
new_pred = model_pipeline.predict(new_data_prepared)
result_df = new_data.copy()
result_df['predicted_next_arrive_time'] = new_pred
result_df['predicted_next_arrive_time'] = result_df['predicted_next_arrive_time'].round(2)

result_df

,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,predicted_next_arrive_time
0,2019-10-21,405136001,7997025,360-1,33.456267,126.55175,제주대학교입구,08시,300.0,제대마을,33.457724,126.554014,37.27
